# SARIMA tuning
Fit on **train**, score MAE on **calibration** (multi-step). Test is never touched here.
Best params get pasted into `SARIMA_PARAMS` in `config.py`.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))


import itertools, time, warnings
import numpy as np, pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from config import DAILY_SERIES_PATH, TRAIN_END, CALIB_END

warnings.filterwarnings("ignore")

In [2]:
s = pd.read_parquet(DAILY_SERIES_PATH).set_index("ds")["y"].asfreq("D")
assert s.isna().sum() == 0, "gaps in daily series"

train = s[:TRAIN_END]
calib = s[TRAIN_END:CALIB_END].iloc[1:]
print(len(train), len(calib), train.index[-1].date(), calib.index[[0, -1]].date)

852 366 2023-12-31 [datetime.date(2024, 1, 1) datetime.date(2024, 12, 31)]


In [3]:
# Reference: repeat the last 7 train days across the calib horizon
naive = np.tile(train.values[-7:], len(calib) // 7 + 1)[:len(calib)]
naive_mae = float(np.mean(np.abs(calib.values - naive)))
print("weekly seasonal-naive calib MAE:", round(naive_mae))

weekly seasonal-naive calib MAE: 8617


In [4]:
S = 7
grid = itertools.product(range(3), range(2), range(3),   # p, d, q
                         range(2), range(2), range(2))   # P, D, Q
rows = []
for p, d, q, P, D, Q in grid:
    order, sorder = (p, d, q), (P, D, Q, S)
    t = time.time()
    try:
        res = SARIMAX(train, order=order, seasonal_order=sorder,
                      enforce_stationarity=False,
                      enforce_invertibility=False).fit(disp=False, maxiter=200)
        pred = res.forecast(len(calib)).values
        mae = float(np.mean(np.abs(calib.values - pred)))
        aic = res.aic
    except Exception:
        mae, aic = np.nan, np.nan
    rows.append(dict(order=order, seasonal_order=sorder, aic=aic,
                     calib_mae=mae, secs=round(time.time() - t, 1)))

results = pd.DataFrame(rows).sort_values("calib_mae").reset_index(drop=True)
results.head(10)

,order,seasonal_order,aic,calib_mae,secs
0,"(1, 0, 0)","(1, 1, 1, 7)",16588.694777,8245.864653,0.2
1,"(1, 0, 0)","(1, 1, 0, 7)",16574.934327,8246.525099,0.4
2,"(1, 0, 1)","(1, 1, 0, 7)",16596.770961,8252.644690,0.2
3,"(1, 0, 0)","(0, 1, 0, 7)",16954.543090,8254.603183,0.1
4,"(1, 0, 1)","(0, 1, 0, 7)",16919.871074,8266.741102,0.1
5,"(2, 0, 0)","(1, 1, 0, 7)",16572.808165,8270.581223,0.1
6,"(1, 0, 2)","(1, 1, 0, 7)",16596.799492,8276.397488,0.2
7,"(1, 0, 2)","(0, 1, 0, 7)",16886.990909,8283.796542,0.1
8,"(2, 0, 0)","(0, 1, 0, 7)",16911.360596,8285.021588,0.1
9,"(1, 0, 1)","(1, 1, 1, 7)",16557.071161,8302.389150,0.2


In [9]:
print("failed fits:", results["calib_mae"].isna().sum(), "of", len(results))
best = results.iloc[0]
print("best calib MAE:", round(best.calib_mae), "| seasonal-naive:", round(naive_mae))
print()
print(f'SARIMA_PARAMS = {{"order": {best.order}, "seasonal_order": {best.seasonal_order}}}')

failed fits: 0 of 144
best calib MAE: 8246 | seasonal-naive: 8617

SARIMA_PARAMS = {"order": (1, 0, 0), "seasonal_order": (1, 1, 1, 7)}


In [6]:
!pip install --quiet pmdarima


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [8]:

import pmdarima as pm

auto = pm.auto_arima(
    train, seasonal=True, m=7,
    max_p=2, max_q=2, max_P=1, max_Q=1, max_d=1, max_D=1,
    information_criterion="aic",
    stepwise=True, suppress_warnings=True, error_action="ignore",
)
print(auto.order, auto.seasonal_order)

res = SARIMAX(train, order=auto.order, seasonal_order=auto.seasonal_order,
              enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred = res.forecast(len(calib)).values
print("auto_arima calib MAE:", round(float(np.mean(np.abs(calib.values - pred)))))

(1, 1, 1) (1, 0, 0, 7)
auto_arima calib MAE: 8791
